# OBR Fase 3 — Segmentação neural da linha

Treina LinhaNet e LR-ASPP somente em treino/validação. O teste de 591 imagens não faz parte do pacote. Selecione uma GPU T4 em **Ambiente de execução → Alterar tipo de ambiente de execução**, execute tudo e entregue somente o ZIP final.

In [ ]:
!test -d /content/OBR || git clone -q https://github.com/DaviBonetto/OBR.git /content/OBR
%cd /content/OBR
!git pull -q --ff-only
!pip install -q -e ".[treinamento]"

In [ ]:
from pathlib import Path

from google.colab import files

print('Selecione o arquivo fase3_dataset_inicial.zip (aprox. 284 MB).')
enviados = files.upload()
assert len(enviados) == 1, 'Envie somente o ZIP do dataset.'
PACOTE = Path.cwd() / next(iter(enviados))
RESULTADOS = Path('/content/resultados_fase3')
assert PACOTE.name == 'fase3_dataset_inicial.zip', PACOTE.name
RESULTADOS.mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib
import shutil

HASH_ESPERADO = '244b4f7b5d15e495fa9059cd8b19f1bb56496c01bb42f6e506bccb7e53bf2c9d'
hash_obtido = hashlib.sha256(PACOTE.read_bytes()).hexdigest()
assert hash_obtido == HASH_ESPERADO, (hash_obtido, HASH_ESPERADO)
DATASET = Path('/content/fase3_dataset_inicial')
if DATASET.exists():
    shutil.rmtree(DATASET)
shutil.unpack_archive(PACOTE, DATASET)
print('Dataset verificado e extraído:', DATASET)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Ative a GPU T4 antes do treinamento completo.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Experimento A — LinhaNet
32 mil parâmetros; candidata principal de baixa latência.

In [ ]:
import subprocess

subprocess.run([
    'obr-treinar-segmentacao', '--dataset', str(DATASET),
    '--saida', str(RESULTADOS / 'linhanet_v1'),
    '--arquitetura', 'linhanet', '--epocas', '100',
    '--lote', '32', '--trabalhadores', '2', '--paciencia', '15',
], check=True)

## Experimento B — LR-ASPP MobileNetV3
Modelo de maior capacidade para comparação e active learning.

In [ ]:
subprocess.run([
    'obr-treinar-segmentacao', '--dataset', str(DATASET),
    '--saida', str(RESULTADOS / 'lraspp_v1'),
    '--arquitetura', 'lraspp_mobilenet_v3_large', '--epocas', '60',
    '--lote', '32', '--trabalhadores', '2', '--paciencia', '10',
], check=True)

In [ ]:
import json

manifestos = {}
for experimento in ('linhanet_v1', 'lraspp_v1'):
    caminho = RESULTADOS / experimento / 'manifesto.json'
    manifestos[experimento] = json.loads(caminho.read_text())
    print(experimento, manifestos[experimento])
vencedor_provisorio = max(manifestos, key=lambda nome: manifestos[nome]['melhor_dice_validacao'])
comparacao = {
    'teste_aberto': False,
    'vencedor_provisorio_por_dice_validacao': vencedor_provisorio,
    'experimentos': manifestos,
}
(RESULTADOS / 'comparacao.json').write_text(
    json.dumps(comparacao, ensure_ascii=False, indent=2), encoding='utf-8'
)

## Entrega única
Gera e baixa automaticamente um ZIP com checkpoints, métricas, históricos, configuração, ambiente e hashes. Esse é o único arquivo que precisa ser enviado de volta ao Codex.

In [ ]:
import datetime
import platform

import torchvision

ENTREGA = Path('/content/OBR_FASE3_RESULTADOS_T4')
if ENTREGA.exists():
    shutil.rmtree(ENTREGA)
shutil.copytree(RESULTADOS, ENTREGA / 'resultados')
shutil.copy2('/content/OBR/configuracoes/treinamento_fase3.toml', ENTREGA)
shutil.copy2('/content/OBR/dados/manifestos/fase3_dataset_inicial.json', ENTREGA)
ambiente = {
    'gerado_utc': datetime.datetime.now(datetime.UTC).isoformat(),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'git_commit': subprocess.check_output(
        ['git', '-C', '/content/OBR', 'rev-parse', 'HEAD'], text=True
    ).strip(),
    'sha256_dataset': hash_obtido,
    'teste_aberto': False,
}
(ENTREGA / 'ambiente.json').write_text(
    json.dumps(ambiente, ensure_ascii=False, indent=2), encoding='utf-8'
)
hashes = {}
for caminho in sorted(ENTREGA.rglob('*')):
    if caminho.is_file():
        hashes[caminho.relative_to(ENTREGA).as_posix()] = hashlib.sha256(
            caminho.read_bytes()
        ).hexdigest()
(ENTREGA / 'sha256_arquivos.json').write_text(
    json.dumps(hashes, indent=2, sort_keys=True), encoding='utf-8'
)
ARQUIVO_FINAL = Path(shutil.make_archive('/content/OBR_FASE3_RESULTADOS_T4', 'zip', ENTREGA))
hash_final = hashlib.sha256(ARQUIVO_FINAL.read_bytes()).hexdigest()
print('ZIP final:', ARQUIVO_FINAL)
print('Tamanho:', ARQUIVO_FINAL.stat().st_size, 'bytes')
print('SHA-256:', hash_final)
files.download(str(ARQUIVO_FINAL))